In [18]:
import pandas as pd
import numpy as np


In [19]:
data=pd.read_csv("spam.csv",encoding="latin1")

In [20]:
data.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [21]:
data.columns

Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='object')

In [22]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


In [23]:
data.shape[0]

5572

In [24]:
data.shape[1]

5

In [25]:
data=data.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'])

In [26]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   v1      5572 non-null   object
 1   v2      5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [27]:
data=data.rename(columns={"v1":"type","v2":"mail"})

In [28]:
data.columns

Index(['type', 'mail'], dtype='object')

In [29]:
def verify(x):
    if x=='ham':
        return 0
    else:
        return 1
data["spamornot"]=data['type'].apply(verify)

In [30]:
data["spamornot"]

0       0
1       0
2       1
3       0
4       0
       ..
5567    1
5568    0
5569    0
5570    0
5571    0
Name: spamornot, Length: 5572, dtype: int64

In [31]:
data.columns

Index(['type', 'mail', 'spamornot'], dtype='object')

In [32]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   type       5572 non-null   object
 1   mail       5572 non-null   object
 2   spamornot  5572 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 130.7+ KB


In [16]:
import re
import nltk
from nltk.corpus import stopwords
#nltk.download('stopwords')
stop_words=set(stopwords.words('english'))
def preprocess_text(text):
    text=text.lower()
    text=re.sub(r'\w', ' ',text)
    text=re.sub(r'\d+', '',text)
    tokens=text.split()
    tokens=[word for word in tokens if word not in stop_words]
    return ' '.join(tokens)
data['clean_text']=data['mail'].apply(preprocess_text)



In [33]:
data.head()

,type,mail,spamornot
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [34]:
import re
import nltk
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # Keep only lowercase letters and spaces
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(tokens)

data['clean_text'] = data['mail'].apply(preprocess_text)

In [35]:
data.head()

,type,mail,spamornot,clean_text
0,ham,"Go until jurong point, crazy.. Available only ...",0,go jurong point crazy available bugis n great ...
1,ham,Ok lar... Joking wif u oni...,0,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1,free entry wkly comp win fa cup final tkts st ...
3,ham,U dun say so early hor... U c already then say...,0,u dun say early hor u c already say
4,ham,"Nah I don't think he goes to usf, he lives aro...",0,nah dont think goes usf lives around though


In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(data['clean_text']).toarray()
y = data['spamornot'].values

In [37]:
data.head()

,type,mail,spamornot,clean_text
0,ham,"Go until jurong point, crazy.. Available only ...",0,go jurong point crazy available bugis n great ...
1,ham,Ok lar... Joking wif u oni...,0,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1,free entry wkly comp win fa cup final tkts st ...
3,ham,U dun say so early hor... U c already then say...,0,u dun say early hor u c already say
4,ham,"Nah I don't think he goes to usf, he lives aro...",0,nah dont think goes usf lives around though


In [38]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [41]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score,classification_report


svc_model = LinearSVC()
svc_model.fit(X_train, y_train)

y_pred_svc = svc_model.predict(X_test)

print("Linear SVC Accuracy:", accuracy_score(y_test, y_pred_svc))
print("\nClassification Report:\n", classification_report(y_test, y_pred_svc))

Linear SVC Accuracy: 0.9766816143497757

Classification Report:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99       965
           1       0.97      0.85      0.91       150

    accuracy                           0.98      1115
   macro avg       0.97      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [42]:
def predict_spam(text):
    cleaned = preprocess_text(text)
    vectorized = tfidf.transform([cleaned]).toarray()
    prediction = model.predict(vectorized)[0]
    return "Spam" if prediction == 1 else "Ham (Not Spam)"

# Example test
print(predict_spam("Congratulations! You've won a free ticket. Call now!"))

NameError: name 'model' is not defined